# 02 — Activation Patching Baseline on Controlled NLI

This notebook runs the **activation patching** baseline for our controlled
lexical-NLI causal-abstraction study.

For each base/source NLI pair, for each `(layer, component, token position)`
site, we:

1. tokenize base and source,
2. run the unwrapped model on each (the *clean* base and source runs),
3. wrap the model in a `pyvene.IntervenableModel` with a single
   `VanillaIntervention` at the requested site,
4. patch the source activation into the base run and read off the
   counterfactual logits,
5. record per-example logit-difference recovery **and** whether the
   patched argmax matches the high-level counterfactual label.

The result is a tidy `pandas.DataFrame` with one row per
`(example, layer, component, position)`, which we then pivot into
layer × position heatmaps (one per component) and save under
`outputs/patching_heatmaps/`.

> Tip: the notebook is sized to run on a laptop CPU in a couple of minutes
> with `distilgpt2`. If you have a GPU, just set `DEVICE = "cuda"` below.

In [ ]:
# Colab bootstrap: clone this project repo, enter it, and install requirements.
# Run this first in a fresh Colab runtime.
import os
import sys
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/aquantumreality/cs221m-das-mqnli.git"
REPO_NAME = "cs221m-das-mqnli"

# Keep Transformers on the PyTorch path and avoid TensorFlow/Keras import issues.
os.environ["USE_TF"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

IN_COLAB = "google.colab" in sys.modules or Path("/content").exists()

if IN_COLAB:
    repo_root = Path("/content") / REPO_NAME
    if not repo_root.exists():
        subprocess.check_call(["git", "clone", REPO_URL, str(repo_root)])
    os.chdir(repo_root)
else:
    # Local/Jupyter fallback: find the existing repository root instead of cloning.
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "requirements.txt").exists() and (candidate / "src").exists():
            repo_root = candidate
            os.chdir(repo_root)
            break
    else:
        raise FileNotFoundError("Could not find repo root with requirements.txt and src/.")

requirements_path = Path("requirements-colab.txt") if Path("requirements-colab.txt").exists() else Path("requirements.txt")
print("Working directory:", Path.cwd())
print("Installing:", requirements_path)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", str(requirements_path)])

# Fallback for environments where pyvene is missing after requirements install.
try:
    import pyvene  # noqa: F401
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "git+https://github.com/stanfordnlp/pyvene.git"])

print("Bootstrap complete. If packages were newly installed, restart the runtime once, then rerun this cell and continue.")

## 1. Setup

In [2]:
import os, sys, pathlib

# Lightweight setup only. No torch / transformers / pyvene imports here.
# This cell should finish almost immediately.
os.environ["USE_TF"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Add the project root (the directory that contains `src/`) to sys.path so
# we can import the project package regardless of where Jupyter was started.
_HERE = pathlib.Path.cwd()
for _p in [_HERE, *_HERE.parents]:
    if (_p / 'src').is_dir():
        PROJECT_ROOT = _p
        break
else:
    raise RuntimeError('Could not locate the project root (no src/ found).')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print('PROJECT_ROOT =', PROJECT_ROOT)

# Colab/GPU defaults for the stronger baseline run.
# Keep MODEL_NAME identical in notebook 03 so the chosen patching site transfers.
MODEL_NAME = 'gpt2'
N_EXAMPLES = 64
LAYERS     = list(range(12))  # gpt2 has 12 transformer blocks
COMPONENTS = ('mlp_output', 'attention_input', 'block_output')
USE_SINGLE_TEMPLATE = True    # keeps token positions stable for fixed-position DAS

OUT_DIR = PROJECT_ROOT / 'outputs' / 'patching_heatmaps'
OUT_DIR.mkdir(parents=True, exist_ok=True)
print('OUT_DIR =', OUT_DIR)

RuntimeError: Could not locate the project root (no src/ found).

## 2. Load model + verbalizer

In [ ]:
# First heavy imports happen here: torch + HuggingFace transformers.
import torch

from src.utils import set_seed
from src.utils.seed import get_device
from src.models import load_causal_lm
from src.metrics import LabelVerbalizer

set_seed(0)
DEVICE = get_device()   # cuda if available, else cpu
print('DEVICE =', DEVICE)

tokenizer, model = load_causal_lm(MODEL_NAME, device=DEVICE)
n_layers = getattr(model.config, 'n_layer', getattr(model.config, 'num_hidden_layers', None))
print(f'Loaded {MODEL_NAME} on {DEVICE}; n_layers = {n_layers}')

verbalizer = LabelVerbalizer.from_tokenizer(
    tokenizer,
    {'entailment': ' yes', 'neutral': ' maybe', 'contradiction': ' no'},
)
print('verbalizer token ids =', verbalizer.token_ids)

DEVICE = mps


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Loaded distilgpt2 on mps; n_layers = 6
verbalizer token ids = {'entailment': 3763, 'neutral': 3863, 'contradiction': 645}


## 3. Build 20 controlled counterfactual NLI pairs

We target the `lexical_relation` intermediate variable in the high-level
causal model. The patch site (in the surface prompt) defaults to the
*hypothesis word* token — see `_VARIABLE_TO_SOURCE_WORD` in
`src.data.counterfactual_pairs`. We also require the counterfactual label
to differ from the base label, so chance-level IIA is meaningfully below
1.0.

In [ ]:
import pandas as pd

from src.data import build_counterfactual_dataset, ID2LABEL
from src.data.nli_templates import DEFAULT_TEMPLATES

TEMPLATES = [DEFAULT_TEMPLATES[0]] if USE_SINGLE_TEMPLATE else DEFAULT_TEMPLATES
print('Using templates:', [t.name for t in TEMPLATES])

dataset = build_counterfactual_dataset(
    tokenizer,
    target_variable='lexical_relation',
    templates=TEMPLATES,
    n_examples=N_EXAMPLES,
    seed=0,
    require_label_change=True,
)
examples = dataset.examples
print(f'Built {len(examples)} counterfactual examples.')

preview = pd.DataFrame([
    {
        'base_prompt':   ex.base.prompt,
        'source_prompt': ex.source.prompt,
        'base_label':    ID2LABEL[ex.base_label_id],
        'source_label':  ID2LABEL[ex.source_label_id],
        'cf_label':      ID2LABEL[ex.counterfactual_label_id],
        'patch_pos':     ex.intervention_pos,
    }
    for ex in examples[:8]
])
preview

Built 20 counterfactual examples.


,base_prompt,source_prompt,base_label,source_label,cf_label,patch_pos
0,I saw a cat yesterday. I saw a cat yesterday. ...,I saw a mammal yesterday. I saw a wolf yesterd...,entailment,neutral,neutral,9
1,I saw a car yesterday. I saw a vehicle yesterd...,I saw a flower yesterday. I saw a rose yesterd...,entailment,neutral,neutral,9
2,I saw a robin yesterday. I saw a bird yesterda...,I saw a animal yesterday. I saw a dog yesterda...,entailment,neutral,neutral,10
3,I saw a bird yesterday. I saw a robin yesterda...,I saw a wolf yesterday. I saw a mammal yesterd...,neutral,entailment,entailment,9
4,A dog is on the table. A car is on the table. ...,A dog is on the table. A animal is on the tabl...,contradiction,entailment,entailment,8
5,A wolf is on the table. A mammal is on the tab...,A cat is on the table. A hammer is on the tabl...,entailment,contradiction,contradiction,8
6,I saw a rose yesterday. I saw a truck yesterda...,I saw a hammer yesterday. I saw a tool yesterd...,contradiction,entailment,entailment,9
7,I saw a robin yesterday. I saw a bird yesterda...,I saw a tool yesterday. I saw a hammer yesterd...,entailment,neutral,neutral,10


## 4. Run the patching sweep

On `distilgpt2` (CPU) with 20 examples, 3 layers, 3 components, and ~10
positions per sequence this takes roughly a minute. On a GPU it's a few
seconds.

In [ ]:
# This is the first pyvene-heavy import. If this import is slow, it should be
# slow here rather than in the first setup cell.
from src.interventions import run_patching_sweep

df = run_patching_sweep(
    model,
    tokenizer,
    examples,
    layers=LAYERS,
    components=COMPONENTS,
    positions='all',
    metric='logit_recovery',
    device=DEVICE,
    verbalizer=verbalizer,
    batch_size=8,
    progress=True,
)
print('rows:', len(df), '  base_accuracy =', round(df.attrs['base_accuracy'], 3))
df.head()

layer x component:   0%|          | 0/9 [00:00<?, ?it/s]

rows: 3060   base_accuracy = 0.6


,example_id,layer,component,position,base_label,source_label,cf_label,base_score,source_score,patched_score,recovery,iia_correct
0,0,0,mlp_output,0,entailment,neutral,neutral,-4.735519,-4.451614,-4.735519,0.0,0
1,1,0,mlp_output,0,entailment,neutral,neutral,-4.640762,-4.555626,-4.640762,0.0,0
2,2,0,mlp_output,0,entailment,neutral,neutral,-4.609379,-4.685444,-4.609379,-0.0,0
3,3,0,mlp_output,0,neutral,entailment,entailment,2.877529,2.885666,2.877529,0.0,1
4,4,0,mlp_output,0,contradiction,entailment,entailment,3.475185,3.744919,3.475185,0.0,1


In [ ]:
# Quick summary: mean recovery + mean IIA per (layer, component).
summary = (
    df.groupby(['component', 'layer'])
      .agg(recovery=('recovery', 'mean'), iia=('iia_correct', 'mean'))
      .round(3)
      .reset_index()
)
summary

,component,layer,recovery,iia
0,attention_input,0,-0.231,0.300
1,attention_input,2,-0.546,0.300
2,attention_input,4,0.045,0.300
3,block_output,0,-2.382,0.294
4,block_output,2,-1.917,0.294
5,block_output,4,-1.646,0.297
6,mlp_output,0,-2.216,0.297
7,mlp_output,2,-0.882,0.300
8,mlp_output,4,-0.141,0.300


## 5. Save heatmaps per component

We save two figures per component:

- `*_recovery.png`  — mean logit-diff recovery, diverging colormap centered at 0.
- `*_iia.png`       — mean IIA (interchange-intervention accuracy), `[0, 1]` viridis.

In [ ]:
from src.viz import save_patching_heatmap_from_df

saved = []
for comp in COMPONENTS:
    rec_path = OUT_DIR / f'{comp}_recovery.png'
    iia_path = OUT_DIR / f'{comp}_iia.png'
    saved.append(save_patching_heatmap_from_df(df, str(rec_path), value_col='recovery',    component=comp, annot=True))
    saved.append(save_patching_heatmap_from_df(df, str(iia_path), value_col='iia_correct', component=comp, annot=True))

for p in saved:
    print('saved', p)

saved /Users/abhiram3001/Downloads/course project stuff/outputs/patching_heatmaps/mlp_output_recovery.png
saved /Users/abhiram3001/Downloads/course project stuff/outputs/patching_heatmaps/mlp_output_iia.png
saved /Users/abhiram3001/Downloads/course project stuff/outputs/patching_heatmaps/attention_input_recovery.png
saved /Users/abhiram3001/Downloads/course project stuff/outputs/patching_heatmaps/attention_input_iia.png
saved /Users/abhiram3001/Downloads/course project stuff/outputs/patching_heatmaps/block_output_recovery.png
saved /Users/abhiram3001/Downloads/course project stuff/outputs/patching_heatmaps/block_output_iia.png


## 6. Persist the raw sweep as CSV

Useful for downstream analyses (per-example breakdowns, statistical tests,
etc.) without re-running the sweep.

In [ ]:
csv_path = PROJECT_ROOT / 'outputs' / 'patching_sweep.csv'
csv_path.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(csv_path, index=False)
print('wrote', csv_path)

wrote /Users/abhiram3001/Downloads/course project stuff/outputs/patching_sweep.csv


## Next steps

- Scale up to all 6 layers and all default components (still cheap with
  small models).
- Swap in `gpt2` or `gpt2-medium` and compare the heatmaps.
- Move from vanilla patching to **DAS**: replace `VanillaIntervention`
  with `RotatedSpaceIntervention` and train the rotation on the
  counterfactual dataset. The peak `(layer, component)` cells from this
  baseline are the natural starting points to localise the DAS subspace.